# ClinicalBridge — Annotated Demo Walkthrough
**COP-3442: Prompt Engineering Capstone | Bahçeşehir University**

This notebook demonstrates ClinicalBridge processing two complete clinical scenarios end-to-end:
- **Scenario 1 — The Missed Medication** (Mehmet Yıldız, 67M)
- **Scenario 3 — The Silent Deterioration** (Hasan Demir, 75M)

Each cell shows one pipeline step with explanations of the prompt engineering decisions behind it.

> All patient data is entirely simulated. This system is not for clinical use.

## Pipeline Architecture

```
RPM Alert Input
      ↓
[1] Alert Triage Agent     — classifies urgency, formulates retrieval queries
      ↓
[2a] EHR Retrieval Agent  ─┐  parallel dispatch
[2b] Anamnesis Agent      ─┤
      ↓                    ┘
[3] Synthesis Agent        — produces Clinical Context Brief (CCB)
      ↓
Clinical Context Brief
```

---
# SCENARIO 1: The Missed Medication
**Mehmet Yıldız, 67M** — sustained high blood pressure over 3 days.

What the system does not yet know: Mehmet stopped his Lisinopril 10 days ago because of a cough. He has also been taking ibuprofen for headaches — which is contraindicated. Neither fact is in his EHR.

In [ ]:
# Step 0: The incoming RPM alert
alert = {
    'patient_id': 'PT001', 'timestamp': '2026-06-01T06:15:00',
    'device_type': 'blood_pressure_monitor', 'measurement_type': 'systolic_bp',
    'value': 162, 'unit': 'mmHg', 'baseline': 138, 'alert_threshold': 155,
    'alert_category': 'elevated_bp',
    'recent_readings': [142, 145, 148, 158, 165, 162],
    'notes': 'Sustained elevated BP over 3 consecutive days. Diastolic also elevated: 104 mmHg.'
}
print(f"Patient: {alert['patient_id']} | Device: {alert['device_type']}")
print(f"Value: {alert['value']} {alert['unit']} | Threshold: {alert['alert_threshold']} | Baseline: {alert['baseline']}")
print(f"Trend (oldest to newest): {' -> '.join(str(r) for r in alert['recent_readings'])}")
print(f"Deviation above baseline: +{alert['value'] - alert['baseline']} mmHg")

### Step 1: Alert Triage Agent

**Key prompt engineering decisions:**
- `temperature=0.0` — deterministic classification; same alert = same urgency every time
- Concrete numerical thresholds in system prompt (not left to model interpretation)
- Dynamic time window: Urgent alerts use 12 months of EHR history
- 9-step chain-of-thought before producing JSON output

In [ ]:
# Triage Agent output (pre-captured from v3 system)
triage = {
    'urgency': 'Urgent',
    'urgency_rationale': 'Systolic BP 162 mmHg sustained over 3 days. Bilateral threshold breach (systolic 162 + diastolic 104). Deviation of 24 mmHg above baseline in a diabetic patient warrants prompt clinical response.',
    'clinical_question': 'Patient PT001 has sustained Stage 2 hypertension. What is their antihypertensive medication history? Have they reported side effects or stopped any medications?',
    'ehr_query_parameters': {'patient_id': 'PT001', 'relevant_conditions': ['hypertension', 'diabetes'], 'relevant_medications': ['ACE inhibitor', 'ARB', 'antihypertensive'], 'relevant_labs': ['creatinine', 'potassium'], 'time_window_months': 12},
    'anamnesis_query_parameters': {'patient_id': 'PT001', 'focus_areas': ['medication_adherence', 'symptoms', 'lifestyle'], 'clinical_question': 'Is patient taking BP medications? Any side effects or self-discontinuation?'},
    'escalate_immediately': False,
    'escalation_modifiers_applied': []
}
print(f"Urgency: {triage['urgency']}")
print(f"Rationale: {triage['urgency_rationale']}")
print(f"Clinical question: {triage['clinical_question']}")
print(f"EHR time window: {triage['ehr_query_parameters']['time_window_months']} months")
print(f"Anamnesis focus: {', '.join(triage['anamnesis_query_parameters']['focus_areas'])}")
print(f"Escalate immediately: {triage['escalate_immediately']}")

### Step 2: Parallel Retrieval

The Orchestrator dispatches EHR and Anamnesis agents **simultaneously** using asyncio. This is why total latency is 21.8s rather than 30s+.

**EHR Agent** (ChromaDB RAG): finds the dry cough documented at two visits — but does not know Lisinopril was stopped.
**Anamnesis Agent**: finds that Lisinopril was stopped 10 days ago for the cough, and ibuprofen is being taken OTC.

In [ ]:
# EHR Retrieval Agent output
ehr = {
    'relevant_diagnoses': [{'icd10': 'I10', 'description': 'Essential hypertension', 'status': 'active'}, {'icd10': 'E11.9', 'description': 'Type 2 diabetes mellitus', 'status': 'active'}],
    'relevant_medications': [{'name': 'Lisinopril', 'dose': '10mg', 'frequency': 'once daily', 'indication': 'Hypertension'}],
    'relevant_labs': [{'test': 'Serum Creatinine', 'value': 1.1, 'unit': 'mg/dL', 'date': '2026-04-10', 'flag': 'normal'}, {'test': 'Serum Potassium', 'value': 4.1, 'unit': 'mEq/L', 'date': '2026-04-10', 'flag': 'normal'}],
    'relevant_visit_notes': [
        {'date': '2026-04-10', 'provider': 'Dr. Susan Okafor', 'excerpt': 'Patient noted a persistent dry cough but attributed it to allergies. Lisinopril continued without change.'},
        {'date': '2025-11-20', 'provider': 'Dr. Susan Okafor', 'excerpt': 'Mentioned cough occasionally but no change to regimen at this time.'}
    ],
    'allergy_alerts': [{'substance': 'Penicillin', 'reaction': 'Rash', 'severity': 'Moderate', 'relevance_to_alert': 'Not directly relevant'}],
    'pattern_flags': [],
    'retrieval_confidence': 0.88,
    'missing_data_flags': ['No renal labs since 2026-04-10', 'No documentation of patient-initiated medication changes in EHR']
}
print('=== EHR RETRIEVAL AGENT ===')
print(f"Confidence: {ehr['retrieval_confidence']}")
print('Diagnoses:', ', '.join(f"{d['description']} ({d['icd10']})" for d in ehr['relevant_diagnoses']))
print('Active medications (per EHR):', ', '.join(f"{m['name']} {m['dose']}" for m in ehr['relevant_medications']))
print('Visit note excerpts:')
for n in ehr['relevant_visit_notes']:
    print(f"  [{n['date']}]: {n['excerpt']}")
print('Missing data flags:')
for f in ehr['missing_data_flags']: print(f'  - {f}')

In [ ]:
# Anamnesis Agent output — the critical cross-source finding
anamnesis = {
    'reporter_type': 'patient',
    'medication_adherence_summary': 'Patient self-discontinued Lisinopril approximately 10 days ago due to a persistent dry cough affecting sleep and causing social embarrassment. Continues Metformin and Aspirin as prescribed. Had intended to contact clinic for alternative but had not yet done so.',
    'medications_taken_as_prescribed': ['Metformin 500mg twice daily', 'Aspirin 81mg once daily'],
    'medications_stopped_by_patient': [{'name': 'Lisinopril', 'dose': '10mg', 'reason': 'Persistent dry cough — affecting sleep and causing social embarrassment', 'date_stopped_approx': '2026-05-22'}],
    'otc_medications': ['Ibuprofen 400mg — taken for headaches over the past 3 days'],
    'symptom_timeline': [
        {'date': '2026-05-28', 'patient_notes': 'Stopped Lisinopril about a week ago. Cough better. Slight headache.'},
        {'date': '2026-05-31', 'patient_notes': 'BP machine showed 165/104 this morning.'},
        {'date': '2026-06-01', 'patient_notes': 'Bad morning. Headache, BP 162/104. Called clinic.'}
    ],
    'family_history_highlights': 'Father had stroke age 71. Brother had myocardial infarction age 58.',
    'adherence_confidence': 'High',
    'sensitivity_flags': []
}
print('=== ANAMNESIS AGENT ===')
print(f"Reporter type: {anamnesis['reporter_type']} | Adherence confidence: {anamnesis['adherence_confidence']}")
print(f"Adherence summary: {anamnesis['medication_adherence_summary']}")
print()
print('MEDICATIONS STOPPED BY PATIENT:')
for m in anamnesis['medications_stopped_by_patient']:
    print(f"  *** {m['name']} {m['dose']} stopped ~{m['date_stopped_approx']}")
    print(f"      Reason: {m['reason']}")
print()
print('OTC MEDICATIONS (not in EHR):')
for m in anamnesis['otc_medications']: print(f'  - {m}')
print()
print('Symptom timeline:')
for e in anamnesis['symptom_timeline']: print(f"  [{e['date']}] {e['patient_notes']}")
print()
print(f"Family history: {anamnesis['family_history_highlights']}")

### Step 3: Synthesis Agent — Clinical Context Brief

**The synthesis happening here:**
- EHR says: Lisinopril active, dry cough noted at 2 visits
- Anamnesis says: Lisinopril stopped 10 days ago because of the cough
- RPM shows: BP rising exactly since the stop date

**Key prompt engineering:** Every claim must cite (EHR), (RPM), or (Anamnesis). Conflicts go in a dedicated field — cannot be silently resolved. Confidence calculated arithmetically.

In [ ]:
# Final Clinical Context Brief
print('=' * 68)
print('  CLINICAL CONTEXT BRIEF  |  Patient: Mehmet Yıldız, 67M (PT001)')
print('=' * 68)
print()
print('URGENCY: Urgent  |  Confidence: 0.87  |  Latency: 21.8s (target <30s)')
print()
print('DATA NOTE: Patient self-reports discontinuing Lisinopril ~10 days ago (Anamnesis).')
print('           EHR medication list does not reflect this. Effective antihypertensive')
print('           coverage may currently be ZERO.')
print()
print('PRIMARY FINDING:')
print('  BP elevation most likely caused by Lisinopril self-discontinuation ~10 days ago')
print('  (Anamnesis), consistent with RPM trend of BP rising from 142-145 to 162-165 mmHg')
print('  (RPM). Dry cough documented at 2 prior visits but not acted upon (EHR: 2025-11-20,')
print('  2026-04-10). Ibuprofen OTC use compounds BP elevation (Anamnesis).')
print()
print('CONFLICT DETECTED:')
print('  EHR: Lisinopril listed as active prescription')
print('  Anamnesis: Patient stopped Lisinopril ~10 days ago')
print('  -> EHR medication list does not reflect actual patient intake')
print()
print('RECOMMENDED ACTIONS:')
actions = [
    ('High', 'Switch to ARB (e.g. Losartan) — equivalent BP control, no cough', 'ACE cough at 2 EHR visits + patient confirmed reason (Anamnesis)'),
    ('High', 'Immediate ibuprofen cessation — contraindicated with hypertension + diabetes', 'NSAID use confirmed (Anamnesis)'),
    ('High', 'Same-day clinical contact to initiate medication transition', 'Stage 2 HTN x3 days with symptoms (RPM + Anamnesis)'),
]
for i, (conf, action, evidence) in enumerate(actions, 1):
    print(f'  {i}. [{conf}] {action}')
    print(f'     Evidence: {evidence}')
print()
print('GAPS: No renal labs since April 2026 — needed before ARB initiation (EHR)')
print()
print('DISCLAIMER: Decision-support only. Not a diagnosis. Clinician review required.')

---
---
# SCENARIO 3: The Silent Deterioration
**Hasan Demir, 75M** — heart failure patient with 13-day progressive weight gain.

**What makes this hard:** Each daily reading was only slightly above the previous one. The danger is only visible across the full trend. Hasan has a **severe NSAID allergy** in his EHR — reaction: fluid retention and worsening heart failure. He has been taking ibuprofen for knee pain and didn't mention it to anyone.

In [ ]:
# Scenario 3 alert
alert3 = {
    'patient_id': 'PT003', 'device_type': 'connected_scale', 'measurement_type': 'body_weight',
    'value': 91.2, 'unit': 'kg', 'baseline': 85.0, 'alert_threshold': 87.0,
    'recent_readings': [85.0, 85.2, 86.1, 86.8, 87.3, 88.0, 88.8, 89.5, 90.1, 90.8, 91.2]
}
readings = alert3['recent_readings']
total_gain = readings[-1] - readings[0]
daily_gain = total_gain / (len(readings) - 1)
print(f"Patient: {alert3['patient_id']} | Device: {alert3['device_type']}")
print(f"Current weight: {alert3['value']} {alert3['unit']}")
print(f"Baseline: {alert3['baseline']} kg | Threshold: {alert3['alert_threshold']} kg")
print(f"Total gain: +{total_gain:.1f} kg over {len(readings)-1} days")
print(f"Average daily gain: {daily_gain:.2f} kg/day — no plateau")
print()
print('13-day weight readings:')
for i, r in enumerate(readings):
    marker = ' <- THRESHOLD CROSSED' if r >= alert3['alert_threshold'] and readings[i-1] < alert3['alert_threshold'] else ''
    print(f'  Day {i:2d}: {r:.1f} kg{marker}')

In [ ]:
# Triage + EHR key findings for Scenario 3
print('TRIAGE OUTPUT:')
print('  Urgency: Critical  (v2 rule: >6kg/14 days + symptoms in HF patient)')
print('  Escalate immediately: True')
print('  EHR time window: 24 months  (Critical + HF patient)')
print()
print('EHR KEY FINDINGS:')
print('  Diagnoses: Heart failure (I50.9), Hypertension (I10), Atrial fibrillation (I48.91)')
print()
print('  *** ALLERGY ALERT ***')
print('  Substance : NSAIDs')
print('  Reaction  : Fluid retention / worsening heart failure')
print('  Severity  : SEVERE')
print('  Relevance : HIGHLY RELEVANT — fluid retention is the presenting concern')
print()
print('  Labs (2026-05-01):')
print('    BNP: 420 pg/mL (H) — already elevated before current episode')
print('    Serum Creatinine: 1.4 mg/dL (H) — impaired renal function')
print()
print('  Pattern flag: BNP elevation + weight above baseline in HF patient')
print('  -> Flagged for clinician attention (clinical significance: physician to determine)')
print()
print('ANAMNESIS KEY FINDINGS:')
print('  OTC medications: Ibuprofen 400mg — 2-3 times this week for knee joint pain')
print('  Symptoms: Bilateral ankle oedema (worsening), exertional dyspnoea on stairs')
print('  Patient: "Last time my ankles swelled like this I ended up in hospital"')
print('  Prior hospitalisation flag (v3): extracted and passed to Synthesis Agent')

In [ ]:
# Scenario 3 Final CCB
print('=' * 68)
print('  CLINICAL CONTEXT BRIEF  |  Patient: Hasan Demir, 75M (PT003)')
print('=' * 68)
print()
print('URGENCY: Critical  |  Confidence: 0.82  |  Latency: 4.2s (escalation)')
print()
print('ALLERGY CONFLICT [data_quality_warning — first section clinician reads]:')
print('  *** Patient self-reports taking ibuprofen 2-3x this week (Anamnesis). ***')
print('  *** EHR documents SEVERE NSAID allergy: fluid retention + worsening HF. ***')
print('  *** This is a direct allergy violation and a likely contributor to fluid overload. ***')
print()
print('PRIMARY FINDING:')
print('  Progressive fluid retention consistent with HF decompensation (RPM trend).')
print('  13-day weight gain + ankle oedema + exertional dyspnoea (Anamnesis) indicates')
print('  clinically significant fluid accumulation. NSAID allergy violation (EHR + Anamnesis)')
print('  is a likely direct contributor to fluid retention.')
print()
print('CONTRIBUTING FACTORS:')
factors = [
    '*** ALLERGY CONFLICT: Ibuprofen (Anamnesis) + NSAID allergy documented: fluid retention/worsening HF (EHR) ***',
    'Progressive weight gain 6.2kg over 13 days, no plateau (RPM)',
    'Bilateral ankle oedema worsening (Anamnesis)',
    'New exertional dyspnoea on stairs (Anamnesis)',
    'BNP already elevated at 420 pg/mL at last visit (EHR)',
    'Prior hospitalisation for ankle oedema mentioned by patient (Anamnesis — v3 extraction)',
]
for f in factors: print(f'  - {f}')
print()
print('RECOMMENDED ACTIONS (ordered by urgency):')
actions = [
    ('High', 'IMMEDIATE ibuprofen cessation — NSAID ALLERGY VIOLATION', 'EHR allergy (SEVERE) + Anamnesis OTC use'),
    ('High', 'Urgent same-day clinical assessment', '6.2kg gain + oedema + dyspnoea + allergy violation (RPM + Anamnesis)'),
    ('High', 'Urgent labs: BNP + renal panel', 'Creatinine already 1.4 mg/dL (EHR) — diuresis requires monitoring'),
    ('Moderate', 'Consider Furosemide dose adjustment — pending renal results', 'Current 40mg may be insufficient (EHR)'),
]
for i, (conf, action, evidence) in enumerate(actions, 1):
    print(f'  {i}. [{conf}] {action}')
    print(f'     Evidence: {evidence}')
print()
print('NOTE: Allergy conflict appears in 3 locations: (1) data_quality_warning, (2) contributing_factors, (3) first recommended action')
print('      This is enforced by the v2 Safety-First Rule in the Synthesis Agent system prompt.')
print()
print('DISCLAIMER: Decision-support only. Not a diagnosis. Clinician review required.')

## Summary

| Scenario | Alert | Key capability |
|---|---|---|
| 1 — Missed Medication | BP trending up | Cross-source synthesis: EHR cough + Anamnesis Lisinopril stop + RPM 10-day rise |
| 3 — Silent Deterioration | Daily weight gain | Trend detection + allergy conflict in 3 required locations |

**In both scenarios, the critical information was split across sources — no single source could produce the CCB alone.**

Scenario 1: EHR shows Lisinopril active → Anamnesis reveals it was stopped → RPM shows BP rising for exactly 10 days.

Scenario 3: EHR has the allergy → Anamnesis has the ibuprofen use → RPM has the fluid accumulation evidence.

---
> **Disclaimer:** This notebook uses pre-captured outputs from a prototype operating on entirely simulated data. Not for clinical use.